# beam_center: commented notebook version

This notebook mirrors the logic in `beam_center/beam_center.py`, but it does **not** drive the existing `LAB:SIM:BEAM_CENTER` soft IOC.

Instead, it runs the beam-centering logic directly in notebook code with gains, tolerance, and timing parameters defined in the cells below.

In [39]:
import time
from p4p.client.thread import Context

ctx = Context("pva")
CAM = "LAB:SIM:CAM01"
MOTX = "LAB:SIM:HMIR"
MOTY = "LAB:SIM:VMIR"
CTRL = "LAB:SIM:BEAM_CENTER"

def unwrap_scalar(value):
    if hasattr(value, "value"):
        value = value.value
    if isinstance(value, dict) and "value" in value:
        value = value["value"]
    return value

def get_scalar(pv):
    return unwrap_scalar(ctx.get(pv, timeout=2.0))

def put_scalar(pv, value, wait=True):
    ctx.put(pv, value, wait=wait, timeout=5.0)

def read_beam_centroid(camera_prefix):
    return (
        float(get_scalar(f"{camera_prefix}:Stats1:CentroidX_RBV")),
        float(get_scalar(f"{camera_prefix}:Stats1:CentroidY_RBV")),
    )

def read_overlay_center(camera_prefix):
    px = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionX_RBV"))
    py = float(get_scalar(f"{camera_prefix}:Overlay1:1:PositionY_RBV"))
    sx = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeX_RBV"))
    sy = float(get_scalar(f"{camera_prefix}:Overlay1:1:SizeY_RBV"))
    return px + sx / 2.0, py + sy / 2.0

In [40]:
# Inspect the current beam, target, and motor state before running the notebook loop.
{
    "motors": {
        "HMIR_RBV": float(get_scalar(f"{MOTX}.RBV")),
        "VMIR_RBV": float(get_scalar(f"{MOTY}.RBV")),
    },
    "beam": read_beam_centroid(CAM),
    "target": read_overlay_center(CAM),
}

{'motors': {'HMIR_RBV': 2.99, 'VMIR_RBV': 3.41},
 'beam': (1853.5330412722226, 1153.5330412722226),
 'target': (989.0, 395.0)}

In [41]:
# Notebook-side centering parameters.
# sign_x and sign_y describe motor polarity relative to image error.
# If a positive motor move shifts the beam in the positive image direction, use -1.0.
# If a positive motor move shifts the beam in the negative image direction, use 1.0.
PARAMS = {
    "max_iterations": 12,
    "gain_x": 0.001,
    "gain_y": 0.001,
    "sign_x": 1.0,
    "sign_y": 1.0,
    "tolerance": 5.0,
    "settle_time": 1.0,
}

PARAMS

{'max_iterations': 12,
 'gain_x': 0.001,
 'gain_y': 0.001,
 'sign_x': 1.0,
 'sign_y': 1.0,
 'tolerance': 5.0,
 'settle_time': 1.0}

In [42]:
def wait_motors_done(timeout=10.0):
    deadline = time.time() + timeout
    while time.time() < deadline:
        done_x = float(get_scalar(f"{MOTX}.DMOV"))
        done_y = float(get_scalar(f"{MOTY}.DMOV"))
        if done_x == 1.0 and done_y == 1.0:
            return True
        time.sleep(0.1)
    return False

def move_motor_relative(motor_pv, delta):
    current = float(get_scalar(f"{motor_pv}.RBV"))
    put_scalar(f"{motor_pv}.VAL", current + delta, wait=False)

def run_beam_center(
    max_iterations=20,
    gain_x=0.01,
    gain_y=0.01,
    sign_x=1.0,
    sign_y=1.0,
    tolerance=5.0,
    settle_time=1.0,
):
    put_scalar(f"{CAM}:Stats1:ComputeCentroid", 1)
    time.sleep(0.3)
    records = []
    for iteration in range(max_iterations):
        beam_x, beam_y = read_beam_centroid(CAM)
        target_x, target_y = read_overlay_center(CAM)
        err_x = target_x - beam_x
        err_y = target_y - beam_y
        step_x = err_x * gain_x * sign_x
        step_y = err_y * gain_y * sign_y
        records.append({
            "iteration": iteration,
            "beam": (beam_x, beam_y),
            "target": (target_x, target_y),
            "error": (err_x, err_y),
            "step": (step_x, step_y),
        })
        print(
            f"[{iteration}] beam=({beam_x:.1f}, {beam_y:.1f})  "
            f"target=({target_x:.1f}, {target_y:.1f})  "
            f"err=({err_x:.1f}, {err_y:.1f})  "
            f"step=({step_x:.4f}, {step_y:.4f})"
        )
        if abs(err_x) < tolerance and abs(err_y) < tolerance:
            print(f"Converged in {iteration + 1} iterations")
            break
        move_motor_relative(MOTX, step_x)
        move_motor_relative(MOTY, step_y)
        wait_motors_done(timeout=10.0)
        time.sleep(settle_time)
    return records

records = run_beam_center(**PARAMS)
records[-3:] if records else []

[0] beam=(1854.1, 1154.1)  target=(989.0, 395.0)  err=(-865.1, -759.1)  step=(-0.8651, -0.7591)
[1] beam=(1627.0, 1153.8)  target=(989.0, 395.0)  err=(-638.0, -758.8)  step=(-0.6380, -0.7588)
[2] beam=(1417.0, 1154.1)  target=(989.0, 395.0)  err=(-428.0, -759.1)  step=(-0.4280, -0.7591)
[3] beam=(1276.0, 952.0)  target=(989.0, 395.0)  err=(-287.0, -557.0)  step=(-0.2870, -0.5570)
[4] beam=(1181.0, 769.0)  target=(989.0, 395.0)  err=(-192.0, -374.0)  step=(-0.1920, -0.3740)
[5] beam=(1119.0, 648.0)  target=(989.0, 395.0)  err=(-130.0, -253.0)  step=(-0.1300, -0.2530)
[6] beam=(1076.0, 566.0)  target=(989.0, 395.0)  err=(-87.0, -171.0)  step=(-0.0870, -0.1710)
[7] beam=(1047.0, 510.0)  target=(989.0, 395.0)  err=(-58.0, -115.0)  step=(-0.0580, -0.1150)
[8] beam=(1027.0, 474.0)  target=(989.0, 395.0)  err=(-38.0, -79.0)  step=(-0.0380, -0.0790)
[9] beam=(1014.0, 448.0)  target=(989.0, 395.0)  err=(-25.0, -53.0)  step=(-0.0250, -0.0530)
[10] beam=(1007.0, 431.0)  target=(989.0, 395.0)  err

[{'iteration': 9,
  'beam': (1014.0, 448.0),
  'target': (989.0, 395.0),
  'error': (-25.0, -53.0),
  'step': (-0.025, -0.053)},
 {'iteration': 10,
  'beam': (1007.0, 431.0),
  'target': (989.0, 395.0),
  'error': (-18.0, -36.0),
  'step': (-0.018000000000000002, -0.036000000000000004)},
 {'iteration': 11,
  'beam': (1001.0, 418.0),
  'target': (989.0, 395.0),
  'error': (-12.0, -23.0),
  'step': (-0.012, -0.023)}]